# Cross-domain engineering SVG patch trainingSelect Kernel > Colab > Auto Connect with a **GPU** runtime, then Run All.Trains a LoRA on 8,000 lineage-disjoint SVG edit examples across five domains and scores 100held-out examples before and after. The target is constrained patch JSON, not a redrawn SVG, so themodel never performs coordinate arithmetic.Every gold patch was applied to its source and proved to reproduce the target tree: 10,000 of 10,000.Checkpoints go to Drive, so a recycled runtime resumes instead of restarting.

In [ ]:
import subprocess, sys, torchassert torch.cuda.is_available(), 'Select Kernel > Colab > Auto Connect with a GPU runtime first.'print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# Fetch from GitHub rather than uploading: files.upload() is a browser-only widget that the VS Code# notebook renderer disables, which is what blocked the previous attempt.import pathlib, subprocessR = '/content/SVG'if pathlib.Path(R, '.git').exists():    subprocess.run(['git','-C',R,'fetch','-q','--depth','1','origin','cvpr2027-research-package'], check=True)    subprocess.run(['git','-C',R,'reset','-q','--hard','FETCH_HEAD'], check=True)else:    subprocess.run(['git','clone','-q','--depth','1','-b','cvpr2027-research-package',                    '--filter=blob:none','--sparse',                    'https://github.com/ayushdebnath012/SVG.git', R], check=True)subprocess.run(['git','-C',R,'sparse-checkout','set','cvpr2027/scripts','cvpr2027/src',                'cvpr2027/data/engsvg-crossdomain-edit-v1'], check=True)print(subprocess.run(['git','-C',R,'log','--oneline','-1'], capture_output=True, text=True).stdout)

In [ ]:
!pip -q install 'transformers>=4.48' 'peft>=0.14' 'accelerate>=1.2' svgpathtools==1.7.2!pip uninstall -y -q torchao

In [ ]:
from google.colab import drive; drive.mount('/content/drive')

In [ ]:
# Checkpoints and the adapter land on Drive so a recycled VM resumes rather than restarting.!cd /content/SVG/cvpr2027 && PYTHONPATH=src:scripts python scripts/train_crossdomain_svg_patcher.py \    --data data/engsvg-crossdomain-edit-v1 \    --out /content/drive/MyDrive/engsvg-crossdomain-run \    --epochs 1 --eval-count 100 --max-length 4096 --max-new-tokens 1400

In [ ]:
import json, pathlibs = json.loads(pathlib.Path('/content/drive/MyDrive/engsvg-crossdomain-run/summary.json').read_text())print(json.dumps(s, indent=2)[:3000])